# exp139_exp092_exp098_small_rank_slot_merge inference

Kaggle inference run for the selected exp139 saved booster ensemble after train-side review. Normal notebook execution sees the exposed visible test only.

## 1. Setup and configuration

In [ ]:
from pathlib import Path

from settings import ExperimentPaths, get_nested, load_config


def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", cfg_get(config, "experiment.route"))
print("Inference mode:", cfg_get(config, "inference.mode"))
print("Selected variant:", cfg_get(config, "inference.selected_variant"))
print("Selected mode:", cfg_get(config, "inference.selected_mode"))
print("Selected model:", cfg_get(config, "inference.selected_model"))
print("Data dir:", paths.raw_data_dir)
print("Test dir:", paths.test_data_dir)
print("Sample submission:", paths.sample_submission_path)
print("Submission path:", paths.submission_path)
print("Artifacts dir:", paths.artifacts_dir)


## 2. Input checks

In [ ]:
selected_variant = cfg_get(config, "inference.selected_variant")
selected_mode = cfg_get(config, "inference.selected_mode")
selected_model = cfg_get(config, "inference.selected_model")
if not selected_variant:
    raise RuntimeError("inference.selected_variant is required")
if not selected_mode:
    raise RuntimeError("inference.selected_mode is required")
if not selected_model:
    raise RuntimeError("inference.selected_model is required")

for label, path in [
    ("raw data", paths.raw_data_dir),
    ("test data", paths.test_data_dir),
    ("sample submission", paths.sample_submission_path),
]:
    if not Path(path).exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    print(label, path)


## 3. Saved model inference

In [ ]:
from exp092_exp098_small_rank_slot_merge import run_saved_model_inference

summary = run_saved_model_inference(
    output_dir=paths.artifacts_dir,
    submission_path=paths.submission_path,
    sample_submission_path=paths.sample_submission_path,
    data_dir=paths.raw_data_dir,
    test_dir=paths.test_data_dir,
    projection_config=cfg_get(config, "model.u_projection", {}),
    rank_slot_config=cfg_get(config, "model.rank_slot", {}),
    variant_name=selected_variant,
    mode_name=selected_mode,
    model_name=selected_model,
    submission_target_column=cfg_get(config, "data.submission_target_column", "tvt"),
    n_jobs=cfg_get(config, "runtime.num_workers", None),
    pf_seeds=None,
    pf_particles=None,
    fast=False,
    use_gpu="auto",
)
summary


## 4. Output summary

In [ ]:
import json

print(json.dumps(summary["metrics"], indent=2, sort_keys=True))
print("submission:", summary["artifacts"]["submission"])
print("summary:", summary["artifacts"]["summary"])
